# Download RAW da B3

Notebook para baixar novos arquivos da B3 direto para a camada raw, sem construir tabelas trusted.

Arquivos do projeto já preparados para isso:
- `src/ml_ettj26/extractors/b3_pregao_raw.py`
- `src/ml_ettj26/extractors/b3_pregao_specs.py`
- `src/ml_ettj26/utils/io/http.py`
- `src/ml_ettj26/utils/io/storage.py`
- `scripts/run_b3_pregao_raw_backfill.py`

Neste notebook, você só precisa indicar:
- `dataset_key`: `"PriceReport"` ou `"DerivativesMarket-SwapMarketRates"`
- `start_day`
- `end_day`
- opcionalmente `strict`, `timeout_sec`, `max_retries` e `backoff_sec`


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Nao encontrei a raiz do projeto acima de {current}")

PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_PATH:", SRC_PATH)


PROJECT_ROOT: C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26
SRC_PATH: C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\src


In [2]:
from datetime import date

from ml_ettj26.utils.io.http import HTTPConfig, RequestsTransport
from ml_ettj26.utils.io.storage import LocalFileStorage
from ml_ettj26.extractors.b3_pregao_raw import B3PregaoDailyRawExtractor, B3PregaoHttpConfig
from ml_ettj26.extractors.b3_pregao_specs import PRICE_REPORT, SWAP_MARKET_RATES


## Parâmetros

Preencha apenas estes campos antes de rodar.

In [3]:
dataset_key = "PriceReport"
# dataset_key = "DerivativesMarket-SwapMarketRates"

start_day = date(2026, 2, 1)
end_day = date(2026, 3, 22)

strict = False
timeout_sec = 30
max_retries = 4
backoff_sec = 1.0


In [8]:
DATASET_MAP = {
    "PriceReport": PRICE_REPORT,
    "DerivativesMarket-SwapMarketRates": SWAP_MARKET_RATES,
}

if dataset_key not in DATASET_MAP:
    raise ValueError(f"dataset_key invalido: {dataset_key}")

dataset = DATASET_MAP[dataset_key]

transport = RequestsTransport(
    HTTPConfig(
        timeout_sec=timeout_sec,
        max_retries=max_retries,
        backoff_sec=backoff_sec,
    )
)

# Base '.' + caminho absoluto retornado pelo extractor -> salva dentro do projeto.
storage = LocalFileStorage(".")

extractor = B3PregaoDailyRawExtractor(
    transport=transport,
    storage=storage,
    dataset=SWAP_MARKET_RATES,
    cfg=B3PregaoHttpConfig(project_root=PROJECT_ROOT, strict=strict),
)

print("Dataset:", dataset.key)
print("Destino:", PROJECT_ROOT / "data" / "01_raw" / "b3" / dataset.key)


Dataset: PriceReport
Destino: C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport


## Download

In [5]:
paths = extractor.fetch_and_store(start_day=start_day, end_day=end_day)

print(f"Arquivos baixados: {len(paths)}")
for path in paths[:10]:
    print(path)

if len(paths) > 10:
    print(f"... e mais {len(paths) - 10} arquivo(s)")


Arquivos baixados: 33
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260202_20260202.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260203_20260203.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260204_20260204.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260205_20260205.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260206_20260206.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260209_20260209.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260210_20260210.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260211_20260211.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\PR260212_20260212.zip
C:\Users\Dell\OneDrive\Documentos\GitHub\ML-ETTJ26\data\01_raw\b3\PriceReport\

In [10]:
from datetime import timedelta

all_paths = []
downloaded_days = []
skipped_days = []
error_days = []

start_day = date(2026, 1, 30)
end_day = date(2026, 3, 22)

current_day = start_day
while current_day <= end_day:
    try:
        paths = extractor.fetch_and_store(day=current_day)

        if paths:
            path = paths[0]
            all_paths.extend(paths)
            downloaded_days.append((current_day, path))
            print(f"OK   {current_day.isoformat()} -> {path}")
        else:
            skipped_days.append(current_day)
            print(f"SKIP {current_day.isoformat()} -> sem arquivo disponivel")

    except Exception as exc:
        error_days.append((current_day, str(exc)))
        print(f"ERRO {current_day.isoformat()} -> {exc}")

    current_day += timedelta(days=1)

print()
print(
    f"Resumo: {len(downloaded_days)} baixado(s), "
    f"{len(skipped_days)} pulado(s), "
    f"{len(error_days)} erro(s)"
)


SKIP 2026-01-30 -> sem arquivo disponivel
SKIP 2026-01-31 -> sem arquivo disponivel
SKIP 2026-02-01 -> sem arquivo disponivel
SKIP 2026-02-02 -> sem arquivo disponivel
SKIP 2026-02-03 -> sem arquivo disponivel
SKIP 2026-02-04 -> sem arquivo disponivel
SKIP 2026-02-05 -> sem arquivo disponivel
SKIP 2026-02-06 -> sem arquivo disponivel
SKIP 2026-02-07 -> sem arquivo disponivel
SKIP 2026-02-08 -> sem arquivo disponivel
SKIP 2026-02-09 -> sem arquivo disponivel
SKIP 2026-02-10 -> sem arquivo disponivel
SKIP 2026-02-11 -> sem arquivo disponivel
SKIP 2026-02-12 -> sem arquivo disponivel
SKIP 2026-02-13 -> sem arquivo disponivel
SKIP 2026-02-14 -> sem arquivo disponivel
SKIP 2026-02-15 -> sem arquivo disponivel
SKIP 2026-02-16 -> sem arquivo disponivel
SKIP 2026-02-17 -> sem arquivo disponivel
SKIP 2026-02-18 -> sem arquivo disponivel
SKIP 2026-02-19 -> sem arquivo disponivel
SKIP 2026-02-20 -> sem arquivo disponivel
SKIP 2026-02-21 -> sem arquivo disponivel
SKIP 2026-02-22 -> sem arquivo dis

## Conferência rápida

In [ ]:
target_dir = PROJECT_ROOT / "data" / "01_raw" / "b3" / dataset.key
saved_files = sorted(target_dir.glob("*.zip"))

print("Pasta:", target_dir)
print("Total de ZIPs na pasta:", len(saved_files))
for file in saved_files[-10:]:
    print(file.name)
